# CUI Reduction Pipeline - Walkthrough

Using query `"knee arthroplasty"` against a store with 5 clinical notes (236 topic groups).
Document side uses Case 1 (asthma patient, 25 phrases, 691 raw CUIs).

Three parts:
1. CUI Reduction (extract → filter → deduplicate)
2. Topic Formation (surviving CUIs → hierarchy → topics)
3. Query Matching (score → rank → return)

---
## Part 1: CUI Reduction

### Stage 1 - Extraction + ConText

We send the phrase to the UMLS extraction API. It does vector similarity search and returns top_k CUIs.

- `TOP_K=6` — was 3, bumped to 6 because short phrases like "pain" had the right CUI at rank 4-5
- `TOP_P=5000` — candidate pool before ranking, diminishing returns past this
- `COMBINED=True` — phrase+context sentence concatenated before embedding, ~15-20% precision gain

Then medspacy ConText checks negation/temporal. Uses the 102 built-in rules (Chapman 2001). We don't add any custom rules — that's the package team's job.

Important: `batch_parse()` tokenizes the full document once and sets all phrases as spans. O(n_tokens) total, not per phrase. Old code appended each phrase to the end of the document — notes ending with "no f/u needed" negated everything.

In [ ]:
# what the API gives us for "knee arthroplasty"
raw = [
    ('C0745532', 0.2178, 'Knee arthroplasty prosthesis placement'),
    ('C1536796', 0.2207, 'Knee arthroplasty (& replacement)'),
    ('C0187960', 0.2235, 'Prosthetic arthroplasty of knee joint'),
    ('C0375841', 0.2247, 'Knee joint replacement'),
    ('C1961789', 0.2288, 'Knee Endoprostheses'),
    ('C5768203', 0.2292, 'Prosthetic arthroplasty of right knee'),
    ('C2004192', 0.2300, 'Other arthroplasty knee joint (& total)'),
    ('C0845658', 0.2316, 'Total arthroplasty of knee, unilateral'),
    ('C4064772', 0.2343, 'Knee arthroplasty with kinetic balance sensor'),
    ('C3873937', 0.2345, 'Knee femur prosthesis'),
    # ... 15 more (mostly ICD10PCS codes)
]

# context result — no negation, no history, nothing
print(f'{len(raw)} shown of 25 total, 0 negated, all survive')

### Stage 2 - SAB Filter (25 → 10)

`ALLOWED_SABS = {"SNOMEDCT_US"}` — UMLS has 200+ vocabularies and tons of duplication. We keep only SNOMED CT US because it gives us one clean hierarchy. ICD10 codes are for billing, not clinical reasoning. If you need meds, add RXNORM. Labs, add LNC.

In [ ]:
# 15 of the 25 are ICD10PCS, ICD10CM, etc — gone
after_sab = [
    ('C0187960', 'Prosthetic arthroplasty of knee joint'),
    ('C0375841', 'Knee joint replacement'),
    ('C5768203', 'Prosthetic arthroplasty of right knee'),
    ('C0845658', 'Total arthroplasty of knee, unilateral'),
    ('C3873937', 'Knee femur prosthesis'),
    ('C5768204', 'Prosthetic arthroplasty of left knee'),
    ('C5568443', 'Arthroplasty of knee planned'),
    ('C0086511', 'Knee Replacement Arthroplasty'),
    ('C0188146', 'Arthroplasty of knee, tibial plateau'),
    ('C5688901', 'Arthroplasty of knee joint using robotic assistance'),
]
print(f'25 → {len(after_sab)} (15 non-SNOMED removed)')

### Stage 3 - Metadata + IC (10 → 10)

Fetch semantic types from BigQuery, look up IC from precomputed dict.

IC formula: `−log((descendants + 1) / total_nodes)` (Resnik 1995, +1 for Laplace smoothing). Precomputed offline for all 2.3M UMLS nodes, so at runtime it's a dict lookup.

All 10 are valid SNOMED procedures. Nothing removed here.

### Stage 4 - Hierarchy Reduction (10 → 8)

Check `graph.predecessors()` — 1 hop UP only. If a parent CUI is also in our set and the child has IC ≥ parent with the same semantic type, drop the parent.

- `ancestor_depth=3` — covers grandparents. Deeper hits root-level stuff.
- We only check 1 hop because multi-hop would compare cousins which is too aggressive.

In [ ]:
# two parent-child pairs found in the surviving set
print('REMOVED: C0086511 (Knee Replacement Arthroplasty)')
print('  KEPT:  C0845658 (Total arthroplasty of knee, unilateral)')
print('  → child is more specific, same semantic type\n')

print('REMOVED: C0375841 (Knee joint replacement)')
print('  KEPT:  C0187960 (Prosthetic arthroplasty of knee joint)')
print('  → child is more specific, same semantic type\n')

print('10 → 8')

### Stage 5 - Embedding Reduction (8 → 5)

Two paths:
- **n ≤ 20**: fast path — just Otsu + structural guards (pairwise dedup)
- **n > 20**: full Louvain — k-NN graph → community detection → silhouette → dedup

We have 8, so fast path. `_LOUVAIN_MIN_CUIS=20` because that equals `k_neighbors`. When n ≤ k, the k-NN graph is fully connected and Louvain just finds noise.

**Otsu threshold** (1979, originally image binarization): try every split point in the sorted distance array, pick the one maximizing between-class variance. Separates "close pairs" from "far pairs". Capped at median distance so we never mark >50% as close.

In [ ]:
import numpy as np

# simulated pairwise distances for 8 CUIs (28 pairs)
close = [0.04, 0.06, 0.07, 0.08, 0.09, 0.10]
far = [0.31, 0.33, 0.35, 0.37, 0.39, 0.41, 0.43, 0.45, 0.47, 0.49,
       0.51, 0.53, 0.55, 0.57, 0.59, 0.61, 0.63, 0.65, 0.67, 0.69, 0.71, 0.73]
dists = np.array(close + far)

# otsu — same as pipeline code
def otsu(values):
    arr = np.sort(values)
    n = len(arr)
    best_t, best_v = arr[0], -1
    cs = np.cumsum(arr)
    total = cs[-1]
    for i in range(1, n):
        if arr[i] == arr[i-1]: continue
        w0, w1 = i/n, 1-i/n
        m0, m1 = cs[i-1]/i, (total-cs[i-1])/(n-i)
        v = w0 * w1 * (m0-m1)**2
        if v > best_v:
            best_v, best_t = v, float(arr[i])
    return best_t

t = min(otsu(dists), float(np.median(dists)))
print(f'otsu: {otsu(dists):.3f}, median cap: {np.median(dists):.3f}, final: {t:.3f}')
print(f'candidate duplicates: {(dists < t).sum()} pairs')

Below threshold ≠ removed. We check 4 guards (cheapest first). ANY guard says "different" → keep both. ALL fail → true duplicate.

1. **Hierarchy edge** (O(1)) — parent-child? keep both
2. **Semantic types** (O(|types|)) — Drug vs Disease? keep both
3. **Ancestor sets** (BFS UP 1-5 hops) — zero shared ancestors? keep both
4. **Sibling leaves** (O(1)) — both leaf nodes under same parent? (left vs right knee) keep both

In [ ]:
# pair 1: true duplicate
print('C0745532 (Knee arthroplasty prosthesis placement)')
print('vs C0187960 (Prosthetic arthroplasty of knee joint)')
print(f'dist=0.04 < {t:.3f}')
print('  guard 1 hier edge? NO')
print('  guard 2 types?     NO (both T061)')
print('  guard 3 ancestors? NO (share Arthroplasty)')
print('  guard 4 siblings?  NO (C0187960 has children)')
print('  → all failed → remove C0745532\n')

# pair 2: guards protect
print('C3873937 (Knee femur prosthesis) — Device T074')
print('vs C0188146 (Arthroplasty of knee, tibial plateau) — Procedure T061')
print(f'dist=0.12 < {t:.3f}')
print('  guard 1 hier edge? NO')
print('  guard 2 types?     YES — Device ≠ Procedure → keep both\n')

print('8 → 5')

In [ ]:
# the 5 survivors
for cui, term, ic in [
    ('C5568443', 'Arthroplasty of knee planned', 14.8),
    ('C5688901', 'Arthroplasty using robotic assistance', 14.8),
    ('C5768203', 'Prosthetic arthroplasty of right knee', 14.8),
    ('C3873937', 'Knee femur prosthesis', 14.8),
    ('C0188146', 'Arthroplasty of knee, tibial plateau', 14.8),
]:
    print(f'{cui}  IC={ic}  {term}')

### When Louvain runs (n > 20)

Didn't happen here — all entities had ≤20 CUIs after hierarchy. But if you had a complex surgical phrase producing 28 SNOMED CUIs:

1. Otsu+guards first → say 28→24
2. k-NN graph: k=20 neighbors per CUI, edge threshold = mean of k-th neighbor distances
3. Louvain (resolution=1.0, seed=42) → say 5 communities
4. Silhouette check: gap > 2× median (Tibshirani 2001) → re-split poor ones at resolution=2.0
5. Singletons merge if sim > mean(within-group) × 0.5
6. Otsu+guards again within each community

### Document side — per entity

Case 1 has 25 phrases. Pipeline runs Stages 2-5 **per phrase**. Metadata/embedding fetches are batched, but reduction is per-entity.

In [ ]:
# from actual logs
ents = [
    ('severe persistent asthma', 28, 24, 19, 13),
    ('allergic rhinitis',        33, 23, 19, 12),
    ('worsening cough',           6,  4,  4,  3),
    ('shortness of breath',      41,  2,  2,  2),  # 39 were ICD10CM
    ('moderate exacerbation',    29, 18, 16,  9),
    ('albuterol',                34,  9,  7,  4),
    ('fluticasone/salmeterol',   18, 10,  7,  4),
    ('montelukast',              32, 14, 11,  7),
    ('prednisone',               20,  6,  5,  3),
    ('FEV1',                     21,  6,  5,  3),
]

print(f'{"phrase":<28} {"raw":>4} {"sab":>4} {"hier":>5} {"emb":>4}')
print('-'*50)
for e in ents:
    print(f'{e[0]:<28} {e[1]:>4} {e[2]:>4} {e[3]:>5} {e[4]:>4}')
print(f'\n138 unique surviving CUIs across all 25 phrases')
print(f'louvain never ran — every entity had ≤20 after hierarchy')

---
## Part 2: Topic Formation

HierarchicalTopicBuilder takes **only CUI codes**. No phrases, no text.

- `max_depth=5` — safety cap. IC gradient does the real stopping
- `IC_GRADIENT_RATIO=0.3` — at hop 3+, stop if parent IC < 30% of current. Catches the cliff where things go broad. Hops 1-2 always followed
- `IC floor = p10(surviving ICs)` — absolute min. Driven by least specific, not bulk
- `max_coverage = max(2, n/√n)` — from IDF lit. Ancestor covering more than this is too common

### Query: 5 CUIs → 1 topic

In [ ]:
# IC floor = p10([14.8]*5) ≈ 14.8
# max_coverage = max(2, 5/√5) = 2

print('''
UMLS hierarchy (BFS UP from 5 surviving CUIs):

✘ Procedure                                    IC=0.8   excluded
│
✘ Surgical procedure on joint                  IC=3.1   excluded
│
✘ Operation on knee                            IC=6.4   excluded
│
├── █ Arthroplasty of knee joint               IC=9.7   CONVERGENCE → Subtopic B
│   │  covers: C5568443, C5688901, C0188146
│   ├── ✔ C5568443  Arthroplasty of knee planned            IC=14.8
│   ├── ✔ C5688901  Arthroplasty using robotic assist       IC=14.8
│   └── ✔ C0188146  Arthroplasty of knee, tibial plateau    IC=14.8
│
└── █ Prosthesis of knee                       IC=10.2  CONVERGENCE → Subtopic A
    │  covers: C3873937, C5768203
    ├── ✔ C5768203  Prosthetic arthroplasty of R knee       IC=14.8
    └── ✔ C3873937  Knee femur prosthesis                   IC=14.8

✘ excluded   █ convergence   ✔ surviving CUI
''')

print('convergence sorted by IC:')
print('  Prosthesis of knee (10.2) → 2 CUIs → Subtopic A')
print('  Arthroplasty of knee joint (9.7) → 3 CUIs → Subtopic B')
print('  Operation on knee (6.4) → rejected, below IC floor')
print()
print('→ 1 topic, 2 subtopics, 5 members, 1 centroid')

### Document: 138 CUIs → 60 topics

Same process but IC floor = p10(138 ICs) ≈ 10.5 — lower because wider IC range. 'Persistent asthma' (IC=10.8) gets to be a grouper here but would be excluded in the query.

In [ ]:
print('''
Topic 8: Exacerbation of Asthma (7 CUIs)

✘ Disorder of respiratory system               IC=3.2   excluded
│
✘ Asthma                                       IC=7.4   excluded (would swallow everything)
│
█ Persistent asthma                            IC=10.8  TOPIC GROUPER
│
├── █ Exacerbation of asthma (C0349790)        IC=12.9  Subtopic A
│   ├── ✔ C1960050  Persistent asthma with exacerbation     IC=14.8
│   ├── ✔ C2887453  Mild Persistent + Acute Exacerbation    IC=14.8
│   └── ✔ C4041140  Acute severe exacerb + rhinitis         IC=14.8
│
├── █ Moderate persistent asthma               IC=12.3  Subtopic B
│   ├── ✔ C2887456  Uncomplicated moderate persistent       IC=14.8
│   └── ✔ C4040110  Acute severe exacerb moderate persist   IC=14.1
│
└── █ Mild persistent asthma                   IC=12.1  Subtopic C
    ├── ✔ C4040055  Mild persistent + allergic rhinitis     IC=14.8
    └── ✔ C4040085  Mild persistent allergic uncontrolled   IC=14.8

Why Asthma (IC=7.4) excluded: below IC floor (10.5). If we used it
as a grouper it would pull in medications, findings, everything.
Persistent asthma (10.8) is just above floor — groups severity variants only.
''')

print('60 topics total, 67 reps, 60 centroids')

---
## Part 3: Query Matching

**Fixed params:**
- `z_score_normalize=True` — auto-weights signals per query. Broad query → vec_sim dominates. Specific → cui_ratio. Can't do that with fixed weights.
- `score_ratio=0.40` — return ≥40% of best. 0.30=noise, 0.60=misses secondaries.
- `score_min=0.01` — structural floor. Catches float artifacts.
- `anc_depth=8` — deeper at search time vs ingestion (3-5). We want ALL relationships.
- Zero-evidence guard: suppress only if ALL three signals are 0.0. No per-signal thresholds — those killed broad queries.

**Adaptive (from this store):**
- `filter_min_ancestor_ic=13.98` — p25 of store ICs. Specific store → high floor.
- `ic_weight_alpha=0.90` — specific query → IC dominates over IDF.
- `ic_prune_ratio=0.711` — tight pruning. p10/p50 of store ICs.
- `sibling_ic_ratio=0.944` — only close siblings. p25/p75.

In [ ]:
# Pass 1: topic gate
# performance optimization, not a hard gate
# if nothing passes → score all groups (fallback)

print('pass 1: query centroid × 236 group centroids')
print('  centroid gate: 1 group clears (case_2)')
print('  CUI overlap rescue: group 71 shares C3873937 → admitted')
print('  → 2/236 groups to pass 2')

In [ ]:
# Pass 2: three signals
# note: doc CUI ancestors precomputed before scoring (fixed a bug where
# the ancestor cache was empty → hierarchy matching silently returned 0)
# also: topic_cui included in matchable set (another bug — the grouping
# ancestor was invisible to the scorer)

print('case_2:')
print('  cui_ratio = 0.38  (3/5 query CUIs matched: C5768203 exact, C0188146 parent-child, C5688901 MICA)')
print('  vec_sim   = high  (both are knee arthroplasty embeddings)')
print('  lin_avg   = 0.617 (strong hierarchy agreement for unmatched CUIs)')

In [ ]:
# Pass 3: z-score
# z = (val - mean) / std per signal, then sum, shift to [0,1]
# highest-variance signal gets most weight automatically
# if all flat → falls back to 0.40/0.30/0.30 weighted raw

print('case_2 normalized: 1.000')

In [ ]:
# Pass 4: relative threshold
best = 1.000
ratio = 0.40
floor = best * ratio

print(f'threshold: {best} × {ratio} = {floor}')
print(f'zero-evidence: nobody has all 3 at 0.0')
print()
print(f'#1  case_2  score=1.000  cui=0.38  lin=0.617')
print(f'    matched: C0188146, C5688901, C5768203')
print()
print(f'1 result. other docs are asthma/diabetes — below {floor}. correct.')

In [ ]:
# summary
print(f'{"step":<22} {"query":>20} {"document":>20}')
print('-'*65)
for s, q, d in [
    ('input',              '"knee arthroplasty"', 'clinical note'),
    ('stage 1: extract',   '25 CUIs',             '691 CUIs'),
    ('stage 2: sab',       '25→10',               '691→~280'),
    ('stage 3: ic',        '10→10',               '~280→~270'),
    ('stage 4: hierarchy', '10→8',                '~270→~209'),
    ('stage 5: embedding', '8→5',                 '209→138'),
    ('stage 6: topics',    '1 topic',             '60 topics'),
    ('pass 1: gate',       '2/236 groups',        '—'),
    ('pass 2: score',      '3 signals',           '—'),
    ('pass 3: z-score',    'normalized',           '—'),
    ('pass 4: threshold',  '1 result',             '—'),
]:
    print(f'{s:<22} {q:>20} {d:>20}')
print(f'\ntotal: 4,417ms')